# 03 · Proyecto final: selección de modelos con tickets reales

**Objetivo empresarial.** Comparar Gemini y un modelo multimodal local usando el mismo contrato,
ground truth y métricas: exactitud, latencia, tokens y costo estimado.

**Ruta en la presentación:** Bloque B (selección/costos) → Bloque D (evaluación y decisión).

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from receipt_validation.config import load_settings

settings = load_settings(ROOT)
print(f"Project root: {ROOT}")

In [ ]:
import json
import pandas as pd
from IPython.display import display

from receipt_validation.clients import ask_model, ask_model_raw, parse_json_content
from receipt_validation.cost import estimate_cost
from receipt_validation.evaluators import evaluate_receipt, summarize_results
from receipt_validation.experiments import run_comparison, save_results
from receipt_validation.io import load_expected_receipts, load_receipt_images
from receipt_validation.prompts import GENERIC_PROMPT, QUESTIONS_PROMPT, build_extraction_prompt
from receipt_validation.schemas import ReceiptExtraction

paths = settings["paths"]
RUN_PROVIDER_CALLS = True
DATASET = "test"  # Use "eval" only for the final unbiased comparison.
labels_file = paths.test_labels_file if DATASET == "test" else paths.eval_labels_file
images_dir = paths.test_receipts_dir if DATASET == "test" else paths.eval_receipts_dir

## 1. Validar datos antes de gastar tokens

La prueba se detiene si el CSV está mal formado o falta una imagen. El conjunto `test` permite
ajustar prompts; `eval` se reserva para la comparación final y evita optimizar contra el examen.

In [ ]:
expected = load_expected_receipts(labels_file)
images = load_receipt_images(images_dir)
# TODO: Compare expected file names against image file names and assert none are missing.
missing_images = []
display(expected.head())

## 2. Experimentos de prompt

Comparamos cuatro variantes: pregunta genérica, extracción zero-shot estructurada, few-shot y
preguntas guiadas. Solo cambia el prompt; imagen, modelo y evaluación permanecen constantes.

In [ ]:
# TODO: Create four variants: generic, structured zero-shot, structured few-shot,
# and structured questions (QUESTIONS_PROMPT).
# Muestra cada prompt COMPLETO con formato (display + Markdown), no solo su longitud.
prompt_variants = {}

### Demo: evaluar los prompts con 5 tickets (modo crudo, sin esquema forzado)

Antes de la matriz completa, evaluamos las cuatro variantes de prompt con 5 tickets para **ver**
cuánto importa el prompt cuando nada más lo rescata.

**Clave:** esta demo usa `ask_model_raw` — **sin** salida estructurada (`responseJsonSchema` /
`response_format`) y **sin** el system prompt de extracción. El prompt de usuario es lo único que
guía al modelo. Con el prompt genérico el modelo tiende a inventar formatos o devolver JSON
malformado; con los estructurados, las reglas y el cierre "return only JSON" garantizan salida
parseable. (La matriz completa y el dashboard sí usan esquema forzado, como en producción.)

Escalera de prompts:

| Variante | Qué agrega |
|---|---|
| `generic` | Solo campos + "respond in JSON" |
| `structured_zero_shot` | + reglas de formato, null, reglas de negocio |
| `structured_few_shot` | + ejemplo de salida |
| `structured_questions` | Preguntas numeradas por campo → cierre exigiendo un solo JSON |

- `DEMO_USE_CLOUD = False` → modelo local (LM Studio). `True` → Gemini en la nube.
- En la nube, el cliente limita automáticamente las peticiones por minuto:
  **15/min** para `gemini-3.1-flash-lite` y **5/min** para cualquier otro modelo
  (5 tickets × 4 prompts = 20 llamadas, ~1.5 min con flash-lite; más con otro modelo).

In [ ]:
RUN_PROMPT_DEMO = True  # Activar solo con llaves/servidor listos.
DEMO_USE_CLOUD = True   # False = LM Studio local | True = Gemini en la nube
DEMO_TICKETS = 5

DEMO_BACKEND = "gemini" if DEMO_USE_CLOUD else "lmstudio"
DEMO_MODEL = (
    settings["default_gemini_model"] if DEMO_USE_CLOUD else settings["default_lmstudio_model"]
)

# TODO 1: Para cada uno de los primeros DEMO_TICKETS tickets y cada variante de
#   prompt_variants, llama ask_model_raw (SIN esquema forzado) con la imagen del ticket.
# TODO 2: Intenta parsear la respuesta con parse_json_content y valida con
#   ReceiptExtraction.model_validate; si falla, registra accuracy 0.0 y el error.
# TODO 3: Evalúa los casos exitosos con evaluate_receipt y guarda por fila:
#   prompt, file_name, accuracy, passed, latency_seconds, error y extraction
#   (el dict de extraction.model_dump() para poder comparar campo por campo).
# TODO 4: Construye un DataFrame, haz un pivot de accuracy (ticket x prompt)
#   y una tabla resumen por prompt ordenada por accuracy.
demo_results = []

### ¿Extrae bien o es ilusión? Valor esperado vs. extraído, campo por campo

La tabla de accuracy resume; esta tabla muestra la evidencia. Por cada ticket: la fila
**esperado** (ground truth del CSV) y una fila por variante de prompt con los valores que
ese JSON trajo realmente. Verde = coincide con lo esperado, rojo = no coincide.


In [ ]:
FIELDS = ["fecha", "folio", "rfc_emisor", "estacion", "moneda", "monto_total"]

# TODO 1: Por cada ticket de demo_rows agrega una fila "esperado" con los valores del CSV
#   y una fila por variante con los valores del dict `extraction` guardado en demo_results.
# TODO 2: Construye un DataFrame y coloréalo: verde si el valor coincide con el esperado
#   (monto_total con tolerancia ±0.05; texto normalizado en minúsculas), rojo si no.
# TODO 3: Observa la columna `fecha`: ¿el mismatch es culpa del prompt o del formato
#   del ground truth?
comparison_rows = []

## 3. Un contrato para todos los modelos

Validar JSON separa “el modelo respondió” de “el sistema recibió datos utilizables”.
Los errores se guardan como resultados; no se pierden silenciosamente.

In [ ]:
SAMPLE_JSON = {
    "fecha": "2025-06-04",
    "folio": "GT6693",
    "rfc_emisor": "GTE941124DN6",
    "estacion": "Gasolinera Teoloyucan S.A. de C.V.",
    "moneda": "MXN",
    "monto_total": 7000.0,
    "productos": [],
    "validation": {"total_matches_products": None, "rfc_format_valid": True,
                   "date_format_valid": True, "issues": []},
}
ReceiptExtraction.model_validate(SAMPLE_JSON).model_dump()

## 4. Ejecutar la matriz experimental

> **Pausa:** estimen primero qué combinación ganará en calidad, costo y latencia.
Mantengan `RUN_PROVIDER_CALLS=False` durante la explicación; actívenlo solo con llaves/servidor listos.

In [ ]:
def run_full_matrix(max_receipts: int | None = None) -> pd.DataFrame:
    # TODO: Build Gemini and LM Studio model configurations.
    # TODO: Compare llm_only and ocr_llm with run_comparison.
    raise NotImplementedError("Complete the sequential comparison runner")

In [ ]:
results_frame = pd.DataFrame()
if RUN_PROVIDER_CALLS:
    results_frame = run_full_matrix()
else:
    print("Provider calls are disabled. Set RUN_PROVIDER_CALLS=True when ready.")

## 5. Tabla, visualización y exportación

La tabla responde qué modelo conviene. El CSV detallado alimenta el dashboard sin volver a gastar
tokens y conserva evidencia por ticket para investigar fallos.

In [ ]:
# TODO: Summarize by pipeline/backend/model and create a ticket-level pivot.
# TODO: Persist detailed results with save_results.
summary = pd.DataFrame()
display(summary)

## 6. Dashboard final

La siguiente celda inicia Streamlit usando el mismo Python 3.12 del kernel. No requiere
JupyterLab ni una terminal externa. El comando equivalente es:

```bash
uv run streamlit run app.py
```

El dashboard permite inspeccionar el ground truth, comparar modelos y revisar el JSON por ticket.

<details><summary><strong>Decisión empresarial para revelar al final</strong></summary>

El modelo ganador no es necesariamente el más preciso. Documenten el umbral mínimo de calidad,
restricciones de privacidad, latencia operativa y costo mensual antes de registrar la decisión.
</details>

In [ ]:
import socket
import subprocess
import sys
import time

DASHBOARD_PORT = 8501

def port_is_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as connection:
        return connection.connect_ex(("127.0.0.1", port)) == 0

if port_is_open(DASHBOARD_PORT):
    print(f"Dashboard already running: http://localhost:{DASHBOARD_PORT}")
else:
    dashboard_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            str(ROOT / "app.py"),
            "--server.headless=true",
            f"--server.port={DASHBOARD_PORT}",
        ],
        cwd=ROOT,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    time.sleep(2)
    print(f"Dashboard started: http://localhost:{DASHBOARD_PORT}")
    print(f"Process id: {dashboard_process.pid}")